# Monte Carlo risk model: South Korean equity portfolio for a euro-based investor

**Question:** how much could a €100,000 KOSPI portfolio lose over 10 trading days, once EUR/KRW currency risk is included?

**Method**
1. Download daily prices for 5 KOSPI stocks and the EUR/KRW rate.
2. Estimate volatilities and the 6×6 covariance matrix (5 stocks + FX) from daily log returns.
3. Cholesky-decompose the covariance (Σ = L·Lᵀ) to turn independent shocks into correlated ones.
4. Simulate 100,000 scenarios under normal and fat-tailed (Student-t) shocks and revalue the portfolio in euros.
5. Report VaR and Expected Shortfall, split the risk by stock and currency, and cross-check against parametric and historical VaR.

**How to run:** open [Google Colab](https://colab.research.google.com) → *File → Upload notebook* → *Runtime → Run all*. Colab downloads the real prices automatically. If the download fails, the notebook falls back to clearly labelled sample data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy import stats
from pathlib import Path

try:
    import yfinance as yf
except ImportError:
    yf = None
    print("yfinance not installed: run  %pip install yfinance  for real data")

## 1. Inputs

In [ ]:
TICKERS = {                       # Yahoo Finance ticker -> name
    "005930.KS": "Samsung Electronics",
    "000660.KS": "SK Hynix",
    "005380.KS": "Hyundai Motor",
    "035420.KS": "NAVER",
    "105560.KS": "KB Financial",
}
FX_TICKER = "EURKRW=X"            # KRW per 1 EUR: a rise = weaker won = loss for a euro investor
WEIGHTS = np.array([0.30, 0.20, 0.20, 0.15, 0.15])   # illustrative allocation across sectors
PORTFOLIO_EUR = 100_000
CONFIDENCE = 0.99
HORIZON = 10                      # trading days (Basel convention)
N_SIMS = 100_000
FAN_DAYS = 63                     # ~3 months, for the path chart only
T_DF = 5                          # degrees of freedom for the fat-tailed scenario
START = "2023-11-01"              # ~3 years of daily data
SEED = 42

assert abs(WEIGHTS.sum() - 1) < 1e-9, "weights must sum to 1"
NAMES = list(TICKERS.values())
FX = "EUR/KRW"

## 2. Data

In [ ]:
def load_real():
    raw = yf.download(list(TICKERS) + [FX_TICKER], start=START, auto_adjust=True, progress=False)["Close"]
    raw[FX_TICKER] = raw[FX_TICKER].ffill()              # FX trades on Korean holidays: carry the last rate
    px = raw.dropna(subset=list(TICKERS)).dropna()       # keep Korean trading days only
    px = px[list(TICKERS) + [FX_TICKER]]
    px.columns = NAMES + [FX]
    if len(px) < 250:
        raise ValueError("not enough data")
    return px

def make_sample(n=750, seed=7):
    # Synthetic prices for testing only. NOT real market data.
    rng = np.random.default_rng(seed)
    vol = np.array([0.30, 0.45, 0.30, 0.35, 0.28, 0.08]) / np.sqrt(252)
    C = np.array([[1, .65, .35, .35, .35, -.30], [.65, 1, .30, .35, .30, -.30],
                  [.35, .30, 1, .30, .40, -.25], [.35, .35, .30, 1, .30, -.20],
                  [.35, .30, .40, .30, 1, -.25], [-.30, -.30, -.25, -.20, -.25, 1]])
    shocks = rng.standard_normal((n, 6)) @ np.linalg.cholesky(C).T * vol
    start = np.array([55000, 180000, 210000, 190000, 80000, 1460.0])
    px = start * np.exp(np.vstack([np.zeros(6), np.cumsum(shocks, axis=0)]))
    idx = pd.bdate_range(end="2026-11-19", periods=n + 1)
    return pd.DataFrame(px, index=idx, columns=NAMES + [FX])

try:
    prices = load_real()
    SAMPLE = False
    print(f"Real data: {len(prices)} trading days, {prices.index[0]:%d %b %Y} – {prices.index[-1]:%d %b %Y}")
except Exception as e:
    prices = make_sample()
    SAMPLE = True
    print("⚠️  Could not download market data (" + type(e).__name__ + "). Using SAMPLE DATA – do not quote these numbers.")
prices.tail()

In [ ]:
# Save the prices for the interactive website (KOSPI Risk Lab): download this file from Colab's Files pane
prices.round(4).to_csv("kospi_prices.csv", index_label="Date", date_format="%Y-%m-%d")
print("Saved kospi_prices.csv" + ("  (sample data)" if SAMPLE else ""))

## 3. Returns, volatility and correlation

In [ ]:
rets = np.log(prices).diff().dropna()                 # daily log returns (stocks in KRW, FX in KRW/EUR)
mu, cov = rets.mean(), rets.cov()
corr = rets.corr()
L = np.linalg.cholesky(cov.values)
assert np.allclose(L @ L.T, cov.values)

# Realised daily portfolio return in EUR: stock value in EUR = KRW price / EURKRW
hist_ret = np.exp(rets[NAMES].values - rets[[FX]].values) @ WEIGHTS - 1

summary = pd.DataFrame({
    "Weight": np.r_[WEIGHTS, np.nan],
    "Annualised volatility": rets.std() * np.sqrt(252),
    "Annualised mean return": mu * 252,
}, index=NAMES + [FX])
summary.style.format("{:.1%}", na_rep="–")

## 4. Monte Carlo simulation

In [ ]:
rng = np.random.default_rng(SEED)

def shocks(n, dist="normal"):
    Z = rng.standard_normal((n, 6))
    if dist == "t":                                      # multivariate Student-t, rescaled to unit variance
        W = rng.chisquare(T_DF, size=(n, 1))
        Z = Z / np.sqrt(W / T_DF) * np.sqrt((T_DF - 2) / T_DF)
    return Z

def simulate(n, h, dist="normal"):
    # h-day correlated log returns; drift set to zero (standard for short-horizon VaR)
    return np.sqrt(h) * shocks(n, dist) @ L.T

def pnl(X):
    s, f = X[:, :5], X[:, 5:]
    eur = PORTFOLIO_EUR * (np.exp(s - f) @ WEIGHTS - 1)   # unhedged: stocks + currency
    local = PORTFOLIO_EUR * (np.exp(s) @ WEIGHTS - 1)     # as if KRW were fully hedged
    return eur, local

def var_es(x, c=CONFIDENCE):
    q = np.quantile(x, 1 - c)
    return -q, -x[x <= q].mean()

X_norm = simulate(N_SIMS, HORIZON, "normal")
X_t = simulate(N_SIMS, HORIZON, "t")
pnl_norm, pnl_local = pnl(X_norm)
pnl_t, _ = pnl(X_t)

VaR, ES = var_es(pnl_norm)
VaR_local, ES_local = var_es(pnl_local)
VaR_t, ES_t = var_es(pnl_t)

# Cross-checks
b = np.r_[WEIGHTS, -1.0]                                  # exposure vector: FX enters with -1
sigma_p = np.sqrt(b @ cov.values @ b)
VaR_param = PORTFOLIO_EUR * stats.norm.ppf(CONFIDENCE) * sigma_p * np.sqrt(HORIZON)
q_h = np.quantile(hist_ret, 1 - CONFIDENCE)
VaR_hist = -q_h * np.sqrt(HORIZON) * PORTFOLIO_EUR
ES_hist = -hist_ret[hist_ret <= q_h].mean() * np.sqrt(HORIZON) * PORTFOLIO_EUR

print(f"{CONFIDENCE:.0%} {HORIZON}-day VaR (MC, normal): €{VaR:,.0f}   ES: €{ES:,.0f}")

## 5. Charts

In [ ]:
# Chart style: one palette, recessive grid, direct labels
INK, INK2, MUTED, GRID, AXIS, SURF = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]
BLUE, ORANGE, RED = "#2a78d6", "#eb6834", "#e34948"
SEQ = {"100": "#cde2fb", "250": "#86b6ef", "450": "#2a78d6", "600": "#184f95"}
plt.rcParams.update({
    "figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF,
    "font.family": "sans-serif", "font.size": 10.5, "text.color": INK,
    "axes.edgecolor": AXIS, "axes.labelcolor": INK2, "axes.grid": True, "axes.axisbelow": True,
    "grid.color": GRID, "grid.linewidth": 0.8, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
})
OUT = Path("charts"); OUT.mkdir(exist_ok=True)
eur = lambda v, p=None: f"€{v:,.0f}" if v >= 0 else f"–€{abs(v):,.0f}"

def frame(fig, ax, title, subtitle, name):
    ax.set_title(title, loc="left", fontsize=14, fontweight="bold", pad=26, color=INK)
    ax.text(0, 1.02, subtitle, transform=ax.transAxes, color=INK2, fontsize=10.5, va="bottom")
    if SAMPLE:
        fig.text(0.99, 0.995, "SAMPLE DATA – not real prices", ha="right", va="top", color=RED, fontsize=9, fontweight="bold")
    fig.savefig(OUT / f"{name}.png")
    plt.show()

In [ ]:
# 5a. Key figures
fig, axes = plt.subplots(1, 4, figsize=(14, 2.6), gridspec_kw={"wspace": 0.06})
tiles = [("Value-at-Risk", VaR, f"{CONFIDENCE:.0%}, {HORIZON} days"),
         ("Expected Shortfall", ES, "avg loss beyond VaR"),
         ("Fat-tailed ES", ES_t, f"Student-t, {T_DF} d.o.f."),
         ("Currency adds", VaR - VaR_local, "to VaR vs. hedged")]
for ax, (lab, val, sub) in zip(axes, tiles):
    ax.axis("off")
    ax.add_patch(plt.Rectangle((0, 0), 1, 1, transform=ax.transAxes, facecolor="white", edgecolor=GRID, lw=1))
    ax.text(0.07, 0.78, lab, color=INK2, fontsize=11, transform=ax.transAxes)
    ax.text(0.07, 0.42, eur(val), color=INK, fontsize=24, fontweight="bold", transform=ax.transAxes)
    ax.text(0.07, 0.08, f"{val / PORTFOLIO_EUR:.1%} of portfolio\n{sub}", color=MUTED, fontsize=9, transform=ax.transAxes)
fig.suptitle(f"€{PORTFOLIO_EUR:,.0f} KOSPI portfolio, euro-based investor", x=0.125, ha="left", fontsize=14, fontweight="bold", y=1.08)
if SAMPLE:
    fig.text(0.99, -0.05, "SAMPLE DATA – not real prices", ha="right", color=RED, fontsize=9, fontweight="bold")
fig.savefig(OUT / "0_key_figures.png"); plt.show()

In [ ]:
# 5b. Price history in euros, rebased to 100
eur_px = prices[NAMES].div(prices[FX], axis=0)
eur_px["KRW (in EUR)"] = 1 / prices[FX]
idx = 100 * eur_px / eur_px.iloc[0]
fig, ax = plt.subplots(figsize=(11, 5.6))
for col, c in zip(idx.columns, SERIES):
    ax.plot(idx.index, idx[col], color=c, lw=2 if col != "KRW (in EUR)" else 2.4,
            ls="-" if col != "KRW (in EUR)" else (0, (4, 2)))
# direct labels at line ends, nudged apart
ends = idx.iloc[-1].sort_values()
span = idx.values.max() - idx.values.min(); pos = []
for v in ends.values:
    pos.append(max(v, pos[-1] + span * 0.045) if pos else v)
for (col, v), y in zip(ends.items(), pos):
    ax.annotate(f"{col}  {v:.0f}", (idx.index[-1], v), xytext=(idx.index[-1] + pd.Timedelta(days=12), y),
                color=INK, fontsize=9.5, va="center", annotation_clip=False,
                arrowprops=dict(arrowstyle="-", color=SERIES[list(idx.columns).index(col)], lw=1))
ax.axhline(100, color=AXIS, lw=1)
ax.set_ylabel("Value in EUR (start = 100)")
ax.legend(idx.columns, ncol=6, loc="upper left", bbox_to_anchor=(0, -0.08), fontsize=9)
ax.margins(x=0)
frame(fig, ax, "Five KOSPI stocks, measured in euros", "Daily prices converted at EUR/KRW. The dashed line is the won itself.", "1_prices_eur")

In [ ]:
# 5c. Correlation matrix
cmap = LinearSegmentedColormap.from_list("div", ["#e34948", "#f0efec", "#2a78d6"])
fig, ax = plt.subplots(figsize=(7.6, 6.4))
im = ax.imshow(corr.values, cmap=cmap, vmin=-1, vmax=1)
labels = [n.replace(" ", "\n") for n in corr.columns]
ax.set_xticks(range(6), labels, fontsize=9); ax.set_yticks(range(6), labels, fontsize=9)
ax.grid(False); [s.set_visible(False) for s in ax.spines.values()]
for i in range(6):
    for j in range(6):
        v = corr.values[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=10,
                color="white" if abs(v) > 0.55 else INK, fontweight="bold" if i == j else None)
ax.set_xticks(np.arange(-.5, 6), minor=True); ax.set_yticks(np.arange(-.5, 6), minor=True)
ax.grid(which="minor", color=SURF, lw=2); ax.tick_params(which="both", length=0)
cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03); cb.outline.set_visible(False); cb.ax.tick_params(colors=MUTED)
frame(fig, ax, "Correlation of daily returns", "Blue = move together, red = move opposite. Feeds the Cholesky factor.", "2_correlation")

In [ ]:
# 5d. Fan chart of simulated portfolio paths
n_paths = 20_000
daily = shocks(n_paths * FAN_DAYS).reshape(n_paths, FAN_DAYS, 6) @ L.T
cum = np.concatenate([np.zeros((n_paths, 1, 6)), np.cumsum(daily, axis=1)], axis=1)
paths = PORTFOLIO_EUR * (np.exp(cum[..., :5] - cum[..., 5:]) @ WEIGHTS)
days = np.arange(FAN_DAYS + 1)
pct = {p: np.percentile(paths, p, axis=0) for p in (1, 5, 25, 50, 75, 95, 99)}

fig, ax = plt.subplots(figsize=(11, 5.6))
for i in range(60):
    ax.plot(days, paths[i], color=MUTED, lw=0.6, alpha=0.35)
ax.fill_between(days, pct[5], pct[95], color=SEQ["100"], lw=0, label="5th–95th percentile")
ax.fill_between(days, pct[25], pct[75], color=SEQ["250"], lw=0, label="25th–75th percentile")
ax.plot(days, pct[50], color=SEQ["600"], lw=2.2, label="Median path")
ax.plot(days, pct[1], color=RED, lw=1.6, ls=(0, (4, 2)), label="1st percentile")
v_h = pct[1][HORIZON]
ax.scatter([HORIZON], [v_h], s=60, color=RED, zorder=5, edgecolor=SURF, lw=2)
ax.annotate(f"Day {HORIZON}: 1-in-100 value {eur(v_h)}", (HORIZON, v_h), xytext=(HORIZON + 3, v_h - PORTFOLIO_EUR * 0.135),
            fontsize=9.5, color=INK, arrowprops=dict(arrowstyle="-", color=RED, lw=1))
ax.yaxis.set_major_formatter(eur); ax.set_xlabel("Trading days ahead"); ax.set_ylabel("Portfolio value")
ax.margins(x=0); ax.legend(loc="upper left", ncol=2, fontsize=9)
frame(fig, ax, f"Where the portfolio could be in {FAN_DAYS} trading days",
      f"{n_paths:,} simulated paths with daily correlated shocks. Grey lines are individual scenarios.", "3_fan_chart")

In [ ]:
# 5e. P&L distribution with VaR and ES
fig, ax = plt.subplots(figsize=(11, 5.6))
bins = np.linspace(np.percentile(pnl_norm, 0.02), np.percentile(pnl_norm, 99.98), 110)
cnt, edges = np.histogram(pnl_norm, bins=bins)
mids, w = (edges[:-1] + edges[1:]) / 2, np.diff(edges)
ax.bar(mids, cnt, width=w * 0.86, color=np.where(edges[1:] <= -VaR, RED, BLUE), lw=0)
ymax = cnt.max()
for x, lab, ls in [(-VaR, f"VaR {eur(VaR)}", (0, (4, 2))), (-ES, f"ES {eur(ES)}", "-")]:
    ax.axvline(x, color=INK, lw=1.4, ls=ls)
    ax.text(x, ymax * (0.92 if "VaR" in lab else 0.78), " " + lab, color=INK, fontsize=10, fontweight="bold", ha="right" if "ES" in lab else "left")
ax.text(-ES, ymax * 0.70, f"worst {1 - CONFIDENCE:.0%} of scenarios ", color=RED, fontsize=9.5, ha="right")
ax.xaxis.set_major_formatter(eur); ax.set_xlabel(f"{HORIZON}-day profit / loss"); ax.set_ylabel("Scenarios")
frame(fig, ax, f"{HORIZON}-day profit and loss across {N_SIMS:,} scenarios",
      f"Red bars are the {1 - CONFIDENCE:.0%} tail. VaR marks where the tail starts; ES is its average.", "4_pnl_distribution")

In [ ]:
# 5f. What the currency does to risk
fig, ax = plt.subplots(figsize=(11, 5.2))
bins = np.linspace(np.percentile(pnl_norm, 0.05), np.percentile(pnl_norm, 99.95), 120)
for data, c, lab, v in [(pnl_local, BLUE, "Stocks only (won hedged)", VaR_local), (pnl_norm, ORANGE, "Stocks + EUR/KRW (unhedged)", VaR)]:
    d, e = np.histogram(data, bins=bins, density=True)
    m = (e[:-1] + e[1:]) / 2
    ax.fill_between(m, d, color=c, alpha=0.18, lw=0); ax.plot(m, d, color=c, lw=2, label=f"{lab}: VaR {eur(v)}")
    ax.axvline(-v, color=c, lw=1.4, ls=(0, (4, 2)))
ax.set_yticks([]); ax.spines["left"].set_visible(False)
ax.xaxis.set_major_formatter(eur); ax.set_xlabel(f"{HORIZON}-day profit / loss"); ax.legend(loc="upper right")
frame(fig, ax, f"The won adds {eur(VaR - VaR_local)} of Value-at-Risk",
      "Same scenarios, with and without the currency. KRW tends to weaken when Korean stocks fall.", "5_currency_effect")

In [ ]:
# 5g. Which positions drive the tail (Euler allocation of Expected Shortfall)
s, f = X_norm[:, :5], X_norm[:, 5:]
stock_pnl = PORTFOLIO_EUR * WEIGHTS * (np.exp(s) - 1)              # each stock in local terms
fx_pnl = pnl_norm - stock_pnl.sum(axis=1)                          # what the currency adds on top
tail = pnl_norm <= -VaR
contrib = pd.Series(np.r_[-stock_pnl[tail].mean(axis=0), -fx_pnl[tail].mean()], index=NAMES + ["EUR/KRW"]).sort_values()
assert np.isclose(contrib.sum(), ES)

fig, ax = plt.subplots(figsize=(10, 4.8))
colors = [ORANGE if k == "EUR/KRW" else BLUE for k in contrib.index]
ax.barh(contrib.index, contrib.values, color=colors, height=0.62)
for i, (k, v) in enumerate(contrib.items()):
    ax.text(v + ES * 0.01, i, f"{eur(v)}  ({v / ES:.0%})", va="center", fontsize=9.5, color=INK)
ax.xaxis.set_major_formatter(eur); ax.grid(axis="y", visible=False); ax.set_xlim(0, contrib.max() * 1.25)
ax.set_xlabel("Contribution to Expected Shortfall")
frame(fig, ax, "Where the tail loss comes from",
      f"Average loss from each position in the worst {1 - CONFIDENCE:.0%} of scenarios. Adds up to ES = {eur(ES)}.", "6_risk_contribution")

In [ ]:
# 5h. Methods compared across confidence levels
levels = [0.95, 0.975, 0.99]
tab = {}
for c in levels:
    q = np.quantile(hist_ret, 1 - c)
    tab[c] = {
        "Monte Carlo (normal)": var_es(pnl_norm, c)[0],
        "Monte Carlo (Student-t)": var_es(pnl_t, c)[0],
        "Parametric": PORTFOLIO_EUR * stats.norm.ppf(c) * sigma_p * np.sqrt(HORIZON),
        "Historical (√h)": -q * np.sqrt(HORIZON) * PORTFOLIO_EUR,
    }
tab = pd.DataFrame(tab)
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(levels)); bw = 0.19
for k, (m, c) in enumerate(zip(tab.index, SERIES)):
    ax.bar(x + (k - 1.5) * (bw + 0.012), tab.loc[m].values, width=bw, color=c, label=m)
ax.set_xticks(x, [f"{c:.1%} confidence" for c in levels]); ax.grid(axis="x", visible=False)
ax.yaxis.set_major_formatter(eur); ax.set_ylabel(f"{HORIZON}-day VaR"); ax.legend(ncol=2, loc="upper left"); ax.set_ylim(0, tab.values.max() * 1.22)
frame(fig, ax, "Four ways to measure the same risk", "Fat tails matter more the further into the tail you look.", "7_methods")
tab.T.style.format("€{:,.0f}")

## 6. Results table

In [ ]:
res = pd.DataFrame({
    "EUR": [VaR, ES, VaR_t, ES_t, VaR_local, VaR - VaR_local, VaR_param, VaR_hist, ES_hist],
}, index=["MC VaR (normal)", "MC ES (normal)", "MC VaR (Student-t)", "MC ES (Student-t)",
          "MC VaR, stocks only (hedged)", "Currency effect on VaR", "Parametric VaR", "Historical VaR (√h)", "Historical ES (√h)"])
res["% of portfolio"] = res["EUR"] / PORTFOLIO_EUR
if SAMPLE: print("⚠️  SAMPLE DATA – do not quote these numbers")
res.style.format({"EUR": "€{:,.0f}", "% of portfolio": "{:.2%}"})

## 7. Assumptions and limitations
- **Normal shocks understate extreme losses.** The Student-t run (5 degrees of freedom) shows how much fatter tails add. Historical simulation is the third view.
- **Volatility and correlation are fixed** over the sample. In a crisis correlations tend to rise. EWMA or GARCH volatility would react faster.
- **Drift is set to zero.** Over 10 days the mean return is tiny and badly estimated.
- **Fat-tailed shocks are scaled by √h** from a single draw, and historical VaR uses √h scaling, which assumes returns are independent from day to day.
- **No rebalancing, dividends or transaction costs.** Weights are an illustrative allocation.
- **Next steps:** backtest the 1-day VaR (Kupiec test), use filtered historical simulation, and stress-test with past KRW sell-offs (2008, 2022).